# 02 — Database Build

Loads the UN SDG Analytics Platform's SQLite database from three source files:

- `Dashboard/data/database_2026.xlsx` — Sustainable Development Report 2026 (SDG Index): raw
  indicator values, normalized scores, composite index, ranks, and indicator metadata (2000-2025,
  193 countries, 124 indicators)
- `data/raw/CLASS_2026_07_15.xlsx` — World Bank country classification (income group)
- `data/raw/API_NY.GDP.PCAP.CD_DS2_en_excel_v2_373157.xls` — World Bank GDP per capita, 1960-2025

Target schema: `sql/01_schema.sql` (5 tables — see project `CLAUDE.md` for design notes).
Output: `data/processed/sdg_analytics.db`.

**Data quirk found while building this:** every sheet in the workbook mixes 15 aggregate rows
(`_OECD`, `_LIC`, `_SIDS`, `_World`, `_G20`, ...) in among the real countries — group averages, not
countries, and they don't line up consistently across sheets (e.g. `_G20` appears in the historical
panel but not the current-year snapshot). These are filtered out everywhere below via an ISO3
format check, keeping only real 3-letter country codes.


In [1]:
import os
import re
import sqlite3
import pandas as pd

DB_XLSX = os.path.join('..', 'Dashboard', 'data', 'database_2026.xlsx')
CLASS_XLSX = os.path.join('..', 'data', 'raw', 'CLASS_2026_07_15.xlsx')
GDP_XLS = os.path.join('..', 'data', 'raw', 'API_NY.GDP.PCAP.CD_DS2_en_excel_v2_373157.xls')
SCHEMA_SQL = os.path.join('..', 'sql', '01_schema.sql')
DB_PATH = os.path.join('..', 'data', 'processed', 'sdg_analytics.db')

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

ISO3_RE = re.compile(r'^[A-Z]{3}$')

def is_iso3(code):
    """True for real 3-letter country codes; filters out the workbook's
    aggregate/group rows (e.g. '_OECD', '_World'), which are not ISO3."""
    return isinstance(code, str) and bool(ISO3_RE.match(code))


## 1. Create schema

Runs `sql/01_schema.sql` against a fresh database file — creates all 5 tables and seeds the static `sdg_goals` reference table (17 rows, hardcoded in the schema file).

In [2]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)  # rebuild from scratch each run — keeps this notebook idempotent

conn = sqlite3.connect(DB_PATH)
with open(SCHEMA_SQL) as f:
    conn.executescript(f.read())
conn.commit()
print('Schema created at', DB_PATH)


Schema created at ..\data\processed\sdg_analytics.db


## 2. `countries`

Base: `SDR2026 Data` sheet (ISO3 code, name, SDR's own 7-region grouping, 2025 population —
the same fields the Power BI dashboard uses, for consistency). Enriched with:
- `income_group` from the World Bank `CLASS` file (left join — a handful of SDG Index
  territories may not appear in the World Bank's 268-economy list, left as NULL)
- `gdp_per_capita` from the World Bank bulk GDP export — most recent non-null year per country
  (coverage varies; some countries' latest data point is a few years old)


In [3]:
sdr = pd.read_excel(DB_XLSX, sheet_name='SDR2026 Data',
                     usecols=['Country Code ISO3', 'country', 'Regions used for the SDR', 'Population in 2025'])
sdr = sdr.rename(columns={
    'Country Code ISO3': 'country_code',
    'country': 'country_name',
    'Regions used for the SDR': 'region',
    'Population in 2025': 'population',
})
sdr = sdr[sdr['country_code'].apply(is_iso3)]

cls = pd.read_excel(CLASS_XLSX, sheet_name='List of economies', usecols=['Code', 'Income group'])
cls = cls.rename(columns={'Code': 'country_code', 'Income group': 'income_group'})

gdp_raw = pd.read_excel(GDP_XLS, sheet_name='Data', header=3)

def is_year_col(c):
    try:
        int(float(c))
        return True
    except (ValueError, TypeError):
        return False

year_cols = [c for c in gdp_raw.columns if is_year_col(c)]
gdp_long = gdp_raw.melt(id_vars=['Country Code'], value_vars=year_cols,
                         var_name='year', value_name='gdp_per_capita')
gdp_long['year'] = gdp_long['year'].apply(lambda c: int(float(c)))
gdp_long = gdp_long.dropna(subset=['gdp_per_capita'])
gdp_latest = gdp_long.sort_values('year').groupby('Country Code', as_index=False).last()
gdp = gdp_latest[['Country Code', 'gdp_per_capita']].rename(columns={'Country Code': 'country_code'})

countries = sdr.merge(cls, on='country_code', how='left').merge(gdp, on='country_code', how='left')
countries['population'] = countries['population'].round().astype('Int64')
countries = countries[['country_code', 'country_name', 'region', 'income_group', 'population', 'gdp_per_capita']]

countries.to_sql('countries', conn, if_exists='append', index=False)
print(len(countries), 'countries loaded')
print('missing income_group:', countries['income_group'].isna().sum())
print('missing gdp_per_capita:', countries['gdp_per_capita'].isna().sum())
countries.head()


193 countries loaded
missing income_group: 0
missing gdp_per_capita: 1


,country_code,country_name,region,income_group,population,gdp_per_capita
0,FIN,Finland,OECD,High income,5622577,56148.580949
1,SWE,Sweden,OECD,High income,10632920,63133.212674
2,DNK,Denmark,OECD,High income,5990921,76970.153522
3,NOR,Norway,OECD,High income,5603541,94594.192957
4,DEU,Germany,OECD,High income,84408921,60496.435082


## 3. `sdg_indicators`

From the `Codebook` sheet — 124 indicators, each mapped to an SDG goal, with the optimum value
(what "goal achieved" looks like) and the lower bound. `target_direction` is derived by comparing
the two: if the optimum is above the lower bound, higher values are better, and vice versa.
`target_code` (UN-style sub-target numbering, e.g. `1.1`) isn't part of this dataset's codebook,
so it stays NULL — this is the SDG Index's own indicator set, not the official UN indicator
framework, and doesn't carry that numbering.


In [4]:
codebook = pd.read_excel(DB_XLSX, sheet_name='Codebook')
cols = codebook.columns
codebook = codebook.rename(columns={
    cols[0]: 'indicator_code',
    cols[1]: 'goal_id',
    cols[7]: 'indicator_name',
    cols[9]: 'target_value',
    cols[12]: 'lower_bound',
})

codebook['target_direction'] = codebook.apply(
    lambda r: 'higher_better' if r['target_value'] > r['lower_bound'] else 'lower_better', axis=1
)
codebook['unit'] = codebook['indicator_name'].str.extract(r'\(([^()]*)\)\s*$')
codebook['target_code'] = None

indicators = codebook[['indicator_code', 'goal_id', 'target_code', 'indicator_name',
                        'unit', 'target_value', 'target_direction']]

indicators.to_sql('sdg_indicators', conn, if_exists='append', index=False)
print(len(indicators), 'indicators loaded')
indicators.head()


123 indicators loaded


,indicator_code,goal_id,target_code,indicator_name,unit,target_value,target_direction
0,sdg1_wpc,1,None,Poverty headcount ratio at $3.00/day (%),%,0.0,lower_better
1,sdg1_lmicpov,1,None,Poverty headcount ratio at $4.20/day (%),%,0.0,lower_better
2,sdg1_oecdpov,1,None,Poverty rate after taxes and transfers (%),%,6.0,lower_better
3,sdg2_undernsh,2,None,Prevalence of undernourishment (%),%,2.5,lower_better
4,sdg2_stunting,2,None,Prevalence of stunting in children under 5 yea...,%,0.0,lower_better


## 4. `sdg_values`

From `All Raw Data` — one row per country per year per indicator, currently stored wide (one
column per indicator code). Reshaped to long format. Rows with no value (the dataset's real,
uneven coverage) are dropped rather than stored as fabricated zeros.


In [5]:
raw = pd.read_excel(DB_XLSX, sheet_name='All Raw Data')
raw = raw[raw['id'].apply(is_iso3)]
indicator_cols = [c for c in raw.columns if c not in ('id', 'country', 'year', 'indexreg')]

values = raw.melt(id_vars=['id', 'year'], value_vars=indicator_cols,
                   var_name='indicator_code', value_name='value')
values = values.rename(columns={'id': 'country_code'})
values = values.dropna(subset=['value'])

valid_codes = set(indicators['indicator_code'])
values = values[values['indicator_code'].isin(valid_codes)]

values['source'] = 'Sustainable Development Report 2026 (SDG Index)'
values['series_code'] = None
values = values[['country_code', 'indicator_code', 'year', 'value', 'source', 'series_code']]

values.to_sql('sdg_values', conn, if_exists='append', index=False)
print(len(values), 'indicator-year-country values loaded')
coverage = values.groupby('country_code')['indicator_code'].nunique()
print('median indicators reported per country:', coverage.median(), '/', len(valid_codes))


293248 indicator-year-country values loaded
median indicators reported per country: 100.0 / 123


## 5. `sdg_index_scores`

Composite index score per country-year from `Backdated SDG Index` (2000-2025), joined with rank
from `Backdated - Ranks over time` (only available from 2015 onward — pre-2015 rows keep a NULL
rank rather than a guessed one). The current 2026 snapshot comes from `SDR2026 Data` since it
isn't part of the backdated panel yet.


In [6]:
bsi = pd.read_excel(DB_XLSX, sheet_name='Backdated SDG Index', usecols=[0, 2, 4])
bsi.columns = ['country_code', 'year', 'sdg_index_score']
bsi = bsi[bsi['country_code'].apply(is_iso3)]

ranks = pd.read_excel(DB_XLSX, sheet_name='Backdated - Ranks over time')
ranks = ranks.rename(columns={ranks.columns[0]: 'country_code', ranks.columns[1]: 'country_name'})
ranks = ranks[ranks['country_code'].apply(is_iso3)]
rank_year_cols = [c for c in ranks.columns if c not in ('country_code', 'country_name')]
ranks_long = ranks.melt(id_vars=['country_code'], value_vars=rank_year_cols,
                         var_name='year_label', value_name='sdg_rank')
ranks_long['year'] = ranks_long['year_label'].str.extract(r'(\d{4})').astype(int)
ranks_long = ranks_long[['country_code', 'year', 'sdg_rank']]

scores = bsi.merge(ranks_long, on=['country_code', 'year'], how='left')

sdr_current = pd.read_excel(DB_XLSX, sheet_name='SDR2026 Data',
                             usecols=['Country Code ISO3', '2026 SDG Index Score', '2026 SDG Index Rank'])
sdr_current = sdr_current.rename(columns={
    'Country Code ISO3': 'country_code',
    '2026 SDG Index Score': 'sdg_index_score',
    '2026 SDG Index Rank': 'sdg_rank',
})
sdr_current = sdr_current[sdr_current['country_code'].apply(is_iso3)]
sdr_current['year'] = 2026
sdr_current = sdr_current[['country_code', 'year', 'sdg_index_score', 'sdg_rank']]

scores_all = pd.concat([scores, sdr_current], ignore_index=True)
scores_all = scores_all.dropna(subset=['sdg_index_score'])

scores_all.to_sql('sdg_index_scores', conn, if_exists='append', index=False)
print(len(scores_all), 'country-year index scores loaded')
print('years covered:', scores_all['year'].min(), '-', scores_all['year'].max())


5187 country-year index scores loaded
years covered: 2000 - 2026


## 6. Sanity checks

Row counts per table, and one join across all five tables to confirm the relational structure
actually holds together (referential integrity, not just five unconnected CSVs in disguise).


In [7]:
conn.commit()

for table in ['sdg_goals', 'countries', 'sdg_indicators', 'sdg_values', 'sdg_index_scores']:
    n = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:20s} {n:>8,} rows')


sdg_goals                  17 rows
countries                 193 rows
sdg_indicators            123 rows
sdg_values            293,248 rows
sdg_index_scores        5,187 rows


In [8]:
check = pd.read_sql_query('''
    SELECT c.country_name, c.region, c.income_group, g.goal_short,
           v.indicator_code, v.year, v.value
    FROM sdg_values v
    JOIN countries c      ON v.country_code = c.country_code
    JOIN sdg_indicators i ON v.indicator_code = i.indicator_code
    JOIN sdg_goals g      ON i.goal_id = g.goal_id
    WHERE c.country_code = 'KEN' AND v.year = 2023
    LIMIT 5
''', conn)
check


,country_name,region,income_group,goal_short,indicator_code,year,value
0,Kenya,Sub-Saharan Africa,Lower middle income,No Poverty,sdg1_wpc,2023,24.311
1,Kenya,Sub-Saharan Africa,Lower middle income,No Poverty,sdg1_lmicpov,2023,36.343
2,Kenya,Sub-Saharan Africa,Lower middle income,Zero Hunger,sdg2_undernsh,2023,36.800
3,Kenya,Sub-Saharan Africa,Lower middle income,Zero Hunger,sdg2_trophic,2023,2.204
4,Kenya,Sub-Saharan Africa,Lower middle income,Zero Hunger,sdg2_crlyld,2023,1.758


In [9]:
conn.close()
print('Done —', DB_PATH)


Done — ..\data\processed\sdg_analytics.db
